# 01 RDD2022 数据审查

纳入日本、印度、捷克、美国、中国摩托车视角和中国无人机视角。挪威子集暂不使用。

In [ ]:
from collections import Counter
from pathlib import Path
import sys
def locate_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'src').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('找不到项目根目录')

PROJECT_ROOT = locate_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from src.constants import CLASS_CODE_TO_NAME, EXCLUDED_SUBSETS, INCLUDED_SUBSETS
from src.data_utils import read_voc_annotation, resolve_subset_directories

SOURCE_ROOT = PROJECT_ROOT / 'datasets' / 'RDD2022'
print('纳入子集：', INCLUDED_SUBSETS)
print('排除子集：', EXCLUDED_SUBSETS)

In [ ]:
rows = []
for subset in INCLUDED_SUBSETS:
    images_dir, annotations_dir = resolve_subset_directories(SOURCE_ROOT / subset)
    images = list(images_dir.rglob('*')) if images_dir.exists() else []
    xml_files = list(annotations_dir.rglob('*.xml')) if annotations_dir.exists() else []
    counts = Counter()
    invalid_xml = 0
    for xml_path in xml_files:
        try:
            for item in read_voc_annotation(xml_path)['boxes']:
                counts[item['class_code']] += 1
        except Exception:
            invalid_xml += 1
    rows.append({'subset': subset, 'images': len([p for p in images if p.is_file()]), 'xml': len(xml_files), 'invalid_xml': invalid_xml, **counts})
rows

In [ ]:
import csv
issues_path = PROJECT_ROOT / 'data' / 'issues.csv'
if issues_path.exists():
    with issues_path.open(encoding='utf-8-sig', newline='') as handle:
        issues = list(csv.DictReader(handle))
    print('问题类型统计：', Counter(row['issue_type'] for row in issues))
else:
    print('尚未生成问题清单，请先执行 02_voc_to_yolo.ipynb')

In [ ]:
sample = None
for subset in INCLUDED_SUBSETS:
    images_dir, annotations_dir = resolve_subset_directories(SOURCE_ROOT / subset)
    for xml_path in annotations_dir.rglob('*.xml') if annotations_dir.exists() else []:
        relative = xml_path.relative_to(annotations_dir)
        candidates = [(images_dir / relative).with_suffix(suffix) for suffix in ['.jpg', '.jpeg', '.png']]
        image_path = next((path for path in candidates if path.exists()), None)
        if image_path:
            sample = (xml_path, image_path)
            break
    if sample:
        break
if sample:
    from PIL import Image, ImageDraw
    import matplotlib.pyplot as plt
    xml_path, image_path = sample
    image = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    for item in read_voc_annotation(xml_path)['boxes']:
        if item['box']:
            draw.rectangle(item['box'], outline='red', width=3)
    plt.figure(figsize=(12, 8))
    plt.imshow(image)
    plt.axis('off')
else:
    print('未找到可展示样例，请检查数据目录')